In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, zscale
import gzip
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from expression_module import (prepare_count_matrix,
                               parse_gmt,
                               calculate_signature_scores,
                               plot_signatures_heatmap,
                               run_deseq2,
                               plot_volcano,
                               plot_boxplot_stats)

# Data loading
Loading pre-cleaned counts dataframe and metadata, mapping ENSEMBL gene names to symbols and converting to log2TPMs.

In [ ]:
metadata = pd.read_csv('../data/metadata_with_predictions.csv', sep = '';'')
log2_tpm, filtered_counts = prepare_count_matrix('../data/counts_matrix.tsv',
                                                 mapping_file,
                                                 filtering_file)

In [ ]:
log2_tpm.head(5)

# Gene Signatures
As part of exploratory data analysis and quality control of used data, we analyzed gene expression patterns for known immune signatures. The analyzed signatures also include those associated with extracellular matrix and angiogenesis. These signatures are expected to be low and are used to confirm that the data used is indeed blood and has not been confounded with tissue.  
We have also added an IFN response signature, which consists of IFN-inducible genes.  
These signatures were calculated for all GSE datasets used in the project. In heatmaps, diagnoses are color-coded: green - healthy, red - SLE, blue - MS, purple - CLE.

In [ ]:
signatures = parse_gmt("gene_signatures.gmt")
ifn_genes = ['IFI27', 'IFIT1', 'IFIT3', 'MX1', 'OAS1', 
             'RSAD2', 'STAT1', 'IFNAR1', 'IFNAR2']
signatures['IFN response'] = ifn_genes
print(f"Signatures loaded: {len(signatures)}")

In [ ]:
unique_gse = metadata['GSE'].unique()
for gse in unique_gse:
    print(gse)
    gse_metadata = metadata[metadata['GSE'] == gse].copy()
    gse_srr = gse_metadata['SRR'].tolist()
    gse_log2_tpm = log2_tpm[gse_srr]
    diagnoses_list = gse_metadata.set_index('SRR').loc[gse_srr, 'diagnosis'].tolist()
    scores_df = calculate_signature_scores(gse_log2_tpm, signatures)
    plot_signatures_heatmap(scores_df, gse, diagnoses_list)

# Differential gene expression
We compared expression of IFN-inducible genes in all samples divided by IFN status and diagnosis. Mann-Whitney U test was used to compare gene expression (log2TPM).

In [ ]:
plot_boxplot_stats(log2_tpm, metadata, ifn_genes,
                   condition_col='diagnosis')

As our classifier revealed IFN-high healthy patients, we wondered whether this is a technical mistake or true biologically significant data. We used DESeq2 to analyze differential gene expression between IFN-high and IFN-low healthy patients. We found that IFN-high healthy patients upregulate IFN-inducible antiviral genes, which could indicate that these patients have recently experienced a viral infection, despite being classified as healthy.  
Notably, these patients come from several datasets, which argues against technical batch effect.

In [ ]:
healthy_metadata = metadata[metadata['diagnosis'] == 'H'].copy()
healthy_samples = healthy_metadata['SRR'].tolist()
healthy_samples = [s for s in healthy_samples if s in filtered_counts.columns]
counts_for_deseq = filtered_counts[healthy_samples].copy()
counts_for_deseq_T = counts_for_deseq.T.astype(int)
healthy_metadata_deseq = healthy_metadata.set_index('SRR').loc[counts_for_deseq_T.index]
res = run_deseq2(counts_for_deseq_T, healthy_metadata_deseq,
                 condition_col='predicted_ifn_status',
                 ref_condition='Low')

In [ ]:
plot_volcano(res, padj_threshold=0.05, lfc_threshold=1)